# K-Nearest Neighbors Implementation

## Overview

In this notebook, I applied **K-Nearest Neighbors** to predict Big Five personality traits (Extraversion, Agreeableness, Conscientiousness, Neuroticism, Openness) from textual life narrative responses (Q1–Q32).

## What is K-Nearest Neighbors?

**KNN** is a non-parametric algorithm that predicts a value by looking at the **K most similar training examples** and averaging their target values (for regression).

For a new narrative, KNN:
1. Computes the distance between that narrative's TF-IDF vector and all training narratives
2. Finds the K closest neighbors
3. Predicts the trait score as the **weighted average** of those K neighbors' scores

### Why KNN for personality prediction?
- Intuitive: people with similar narratives likely have similar personalities
- No assumptions about the functional form of the relationship
- Works as a useful baseline to compare against more complex models
- `weights='distance'` gives closer neighbors more influence

### Caveat
KNN suffers from the **curse of dimensionality** — in high-dimensional TF-IDF space, all points become equidistant. We use TruncatedSVD to reduce dimensions first.

### Pipeline
```
Narrative text (Q columns)
        ↓
TF-IDF Vectorization
        ↓
TruncatedSVD → 100 dense dimensions
        ↓
KNeighborsRegressor (one per trait via MultiOutputRegressor)
        ↓
5-Fold Cross-Validation + Evaluation
```

## 1. Data preprocessing, import required packages

A reusable `KNN` class is defined:
- TF-IDF vectorization
- Truncated SVD dimensionality reduction
- KNN regression training
- Hyperparameter tuning for number of neighbors
- Cross-validation
- Evaluation metrics

This mirrors how scikit-learn's own estimators work, making it easy to reuse across notebooks.

As specified in the random forest module, all narrative columns are combined into a single text string per participant, then apply **TF-IDF** (Term Frequency–Inverse Document Frequency) to convert text into a numeric matrix. 

The key process of supervised learning is for model to learn patterns from training set and evaluate model performance on unseen test set. Thus, data is split into **80% training** and **20% testing**.`random_state=42` ensures reproducibility

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import sys
import os
sys.path.append(os.path.abspath("../../src"))
from ml_models.knn import KNN

# Ignore warnings
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
np.seterr(all="ignore")

print("✓ Imports complete")

# Find repo root automatically
repo_root = Path.cwd().resolve()
while not (repo_root / "BFI_2_life_narative_metadata.json").exists():
    if repo_root == repo_root.parent:
        raise FileNotFoundError("Could not find BFI_2_life_narative_metadata.json")
    repo_root = repo_root.parent

# Load data
with open(repo_root / "BFI_2_life_narative_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

with open(repo_root / "BFI_2_life_narrative.json", "r", encoding="utf-8") as f:
    responses = json.load(f)

df = pd.DataFrame(responses)

# Reverse-code BFI items
reverse_items = metadata["reverse_code_items"]

for item in reverse_items:
    if item in df.columns:
        df[item] = 6 - df[item]

# Big Five trait means
traits = {
    key: value
    for key, value in metadata.items()
    if (
        isinstance(value, list)
        and value
        and isinstance(value[0], str)
        and value[0].startswith("Item")
        and not key.startswith(("Item", "Q", "CWB", "OCB"))
        and key != "reverse_code_items"
    )
}

for trait, items in traits.items():
    available_items = [item for item in items if item in df.columns]
    df[trait] = df[available_items].mean(axis=1)

# CWB and OCB means
cwb_cols = [f"CWB{i}" for i in range(1, 11) if f"CWB{i}" in df.columns]
ocb_cols = [f"OCB{i}" for i in range(1, 11) if f"OCB{i}" in df.columns]

df["CWB"] = df[cwb_cols].mean(axis=1)
df["OCB"] = df[ocb_cols].mean(axis=1)

# Text features
q_cols = [c for c in df.columns if isinstance(c, str) and c.startswith("Q")]

X_text = df[q_cols].copy()
X_all = X_text.fillna("").agg(" ".join, axis=1)

# Targets
big_five = [
    "Extraversion",
    "Agreeableness",
    "Conscientiousness",
    "Neuroticism",
    "Openness",
]

y = df[big_five].copy()

print("✓ Data preprocessing complete")
print("Repo root:", repo_root)
print("df shape:", df.shape)
print("X_text shape:", X_text.shape)
print("X_all shape:", X_all.shape)
print(y.describe().round(3))

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, random_state=42
)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")

✓ Imports complete
✓ Data preprocessing complete
Repo root: /Users/cindy/cmor438_Spring2026/cmor438_Spring2026
df shape: (500, 138)
X_text shape: (500, 32)
X_all shape: (500,)
       Extraversion  Agreeableness  Conscientiousness  Neuroticism  Openness
count       500.000        500.000            500.000      500.000   500.000
mean          3.177          3.786              3.642        2.879     3.833
std           0.761          0.576              0.710        0.846     0.618
min           1.000          2.000              1.083        1.000     1.750
25%           2.667          3.417              3.167        2.333     3.417
50%           3.167          3.833              3.750        2.917     3.833
75%           3.750          4.167              4.083        3.417     4.250
max           5.000          5.000              5.000        4.917     5.000
Training samples : 400
Test samples     : 100


## 2. Tune K with GridSearchCV

The most important hyperparameter in KNN is **K** (number of neighbors):
- **Small K** (e.g., K=1): very sensitive to noise, overfits
- **Large K** (e.g., K=20): smoother predictions, may underfit

We search over K values from 10 to 50 in increments of 5, allowing a potentially different optimal K for each trait.

In [2]:
knn_model = KNN()
best_k = knn_model.tune_k(X_train, y_train)
best_k

/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: invalid value encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:547: RuntimeWarning: invalid value encountered in matmul
  U = Q @ Uhat
/Users/cindy/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:54

{'Extraversion': 35,
 'Agreeableness': 50,
 'Conscientiousness': 20,
 'Neuroticism': 15,
 'Openness': 50}

## 3. Train and test model

In [3]:
knn_model.fit(X_train, y_train)
test_results_knn = knn_model.evaluate(X_test, y_test)
print("Test Set Performance:")
test_results_knn

Test Set Performance:


,R²,MAE,Pearson r
Outcome,,,
Extraversion,0.0157,0.6348,0.1546
Agreeableness,-0.0452,0.4067,0.0055
Conscientiousness,-0.0275,0.6234,0.0399
Neuroticism,-0.0317,0.7432,0.1163
Openness,0.0341,0.5340,0.2962


## 4. 5-fold cross-validation

The previous results seem quite unstable — especially since conscientiousness is usually the most predictable trait — so 5-fold cross-validation was applied to ensure prediction stability.

In [4]:
cv_results_knn = knn_model.cross_validate(X_all.reset_index(drop=True), y.reset_index(drop=True))
print("K-Nearest Neighbors Cross-Validation Results:")
cv_results_knn

K-Nearest Neighbors Cross-Validation Results:


,CV Mean R²,CV Std R²,CV MAE,Pearson r
Outcome,,,,
Extraversion,0.0207,0.0449,0.6050,0.2233
Agreeableness,-0.0255,0.0138,0.4600,0.0046
Conscientiousness,0.0151,0.0364,0.5707,0.1855
Neuroticism,-0.0318,0.0837,0.6801,0.2039
Openness,0.0109,0.0146,0.4897,0.1672


## Results analysis

#### Pearson's r Comparison

| Trait | KNN | Decision Tree | Random Forest | Gradient Boosting |
|---|---:|---:|---:|---:|
| Extraversion | 0.2233 | 0.1297 | **0.2981** | 0.2285 |
| Agreeableness | 0.0046 | -0.0353 | 0.0296 | **0.0710** |
| Conscientiousness | 0.1855 | 0.0737 | **0.2916** | 0.2838 |
| Neuroticism | 0.2039 | 0.1024 | **0.2601** | 0.2213 |
| Openness | 0.1672 | 0.0452 | 0.1396 | **0.2038** |

KNN outperformed the Single Decision Tree on all traits, suggesting that similarity-based prediction was more effective than relying on a single tree split structure. The best tree-based method (random forest) has stronger overall prediction than KNN, indicating that aggregating many trees captured personality-related text patterns better than local distance matching. The prediction performance was on par for Gradient Boosting and KNN, with no single approach dominated across all traits.

This pattern shows that in text data:

- KNN can capture local similarity between narratives
- But sparse high-dimensional text features often make distance measures noisy
- Ensemble methods are usually more robust to this complexity